# Capítulo 7: Construir Aplicações de Chat
## Início Rápido da API Microsoft Foundry Models

Este notebook é adaptado do [Repositório de Exemplos Azure OpenAI](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) que inclui notebooks que acedem a serviços [Azure OpenAI](notebook-azure-openai.ipynb).

> **Nota:** GitHub Models será descontinuado no final de julho de 2026. Este notebook agora direciona para os [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst), que oferece o mesmo catálogo de modelos gratuitos para experimentar e a experiência do Azure AI Inference SDK.


# Visão geral  
"Modelos de linguagem grande são funções que mapeiam texto para texto. Dada uma cadeia de texto de entrada, um modelo de linguagem grande tenta prever o texto que virá a seguir"(1). Este caderno "quickstart" irá introduzir os utilizadores a conceitos de alto nível sobre LLM, requisitos essenciais de pacotes para começar com AML, uma introdução suave ao design de prompts, e vários exemplos curtos de diferentes casos de uso. 


## Índice  

[Visão Geral](#overview)  
[Como usar o Serviço OpenAI](#how-to-use-openai-service)  
[1. Criar o seu Serviço OpenAI](#1.-creating-your-openai-service)  
[2. Instalação](#2.-installation)    
[3. Credenciais](#3.-credentials)  

[Casos de Uso](#use-cases)    
[1. Resumir Texto](#1.-summarize-text)  
[2. Classificar Texto](#2.-classify-text)  
[3. Gerar Novos Nomes de Produto](#3.-generate-new-product-names)  
[4. Ajustar um Classificador](#4.fine-tune-a-classifier)  

[Referências](#references)


### Construa o seu primeiro prompt  
Este pequeno exercício fornecerá uma introdução básica para submeter prompts a um modelo nos Microsoft Foundry Models para uma tarefa simples de "resumo".


**Passos**:  
1. Instale a biblioteca `azure-ai-inference` no seu ambiente Python, caso ainda não o tenha feito.  
2. Carregue as bibliotecas auxiliares padrão e configure as suas credenciais dos Microsoft Foundry Models.  
3. Escolha um modelo para a sua tarefa  
4. Crie um prompt simples para o modelo  
5. Submeta o seu pedido à API do modelo!


### 1. Instalar `azure-ai-inference`


In [ ]:
%pip install azure-ai-inference

### 2. Importar bibliotecas auxiliares e instanciar credenciais


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. Encontrar o modelo certo  
Modelos como o GPT-4o e o GPT-4o mini conseguem compreender e gerar linguagem natural, e estão disponíveis no catálogo Microsoft Foundry Models juntamente com modelos da Meta, Mistral, Cohere e Microsoft.


In [ ]:
# Select a general purpose chat model
model_name = "gpt-5-mini"


## 4. Criação de Prompts  

"A magia dos grandes modelos de linguagem é que, ao serem treinados para minimizar este erro de previsão sobre vastas quantidades de texto, os modelos acabam por aprender conceitos úteis para estas previsões. Por exemplo, aprendem conceitos como"(1):

* como soletrar
* como funciona a gramática
* como parafrasear
* como responder a perguntas
* como manter uma conversa
* como escrever em várias línguas
* como programar
* etc.

#### Como controlar um grande modelo de linguagem  
"De todas as entradas para um grande modelo de linguagem, de longe a mais influente é o prompt de texto(1).

Grandes modelos de linguagem podem ser solicitados a produzir resultados de algumas formas:

Instrução: Diga ao modelo o que quer
Complemento: Induza o modelo a completar o início do que deseja
Demonstração: Mostre ao modelo o que quer, com:
Alguns exemplos no prompt
Centenas ou milhares de exemplos num conjunto de dados de treino para ajuste fino"



#### Existem três directrizes básicas para criar prompts:

**Mostre e diga**. Torne claro o que quer, seja através de instruções, exemplos, ou uma combinação dos dois. Se quer que o modelo ordene uma lista de itens por ordem alfabética ou classifique um parágrafo por sentimento, mostre-lhe que é isso que quer.

**Forneça dados de qualidade**. Se está a tentar construir um classificador ou fazer o modelo seguir um padrão, assegure-se de que existem exemplos suficientes. Certifique-se de rever os seus exemplos — o modelo geralmente é inteligente o suficiente para perceber erros ortográficos básicos e dar uma resposta, mas também pode assumir que isso é intencional e isso pode afectar a resposta.

**Verifique as suas configurações.** As configurações temperature e top_p controlam o quão determinístico o modelo é ao gerar uma resposta. Se está a pedir uma resposta onde há apenas uma resposta correcta, então deve definir estes valores mais baixos. Se procura respostas mais diversas, pode defini-los mais altos. O erro número um que as pessoas cometem com estas configurações é assumir que são controlos de "esperteza" ou "criatividade".


Fonte: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. Submeter!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### Repetir a mesma chamada, como é que os resultados se comparam?


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## Resumir Texto  
#### Desafio  
Resuma o texto adicionando um 'tl;dr:' no final de um trecho de texto. Repare como o modelo compreende como executar várias tarefas sem instruções adicionais. Pode experimentar com prompts mais descritivos do que tl;dr para modificar o comportamento do modelo e personalizar o resumo que recebe(3).  

Trabalhos recentes demonstraram ganhos substanciais em muitas tarefas e benchmarks de PLN ao treinar previamente num grande corpus de texto seguido de ajuste fino numa tarefa específica. Embora tipicamente agnóstico em relação à tarefa na arquitetura, este método ainda requer conjuntos de dados de ajuste fino específicos da tarefa, com milhares ou dezenas de milhares de exemplos. Em contraste, os humanos geralmente conseguem realizar uma nova tarefa linguística a partir de apenas alguns exemplos ou de instruções simples — algo com que os sistemas atuais de PLN ainda lutam bastante. Aqui mostramos que aumentar a escala dos modelos de linguagem melhora bastante o desempenho agnóstico da tarefa com poucos exemplos, por vezes até alcançando competitividade com métodos de ajuste fino de última geração.  



Tl;dr  


# Exercícios para vários casos de uso  
1. Resumir Texto  
2. Classificar Texto  
3. Gerar Novos Nomes de Produto


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Classificar Texto  
#### Desafio  
Classifique itens em categorias fornecidas no momento da inferência. No exemplo a seguir, fornecemos tanto as categorias quanto o texto a classificar no prompt (*playground_reference). 

Pedido do Cliente: Olá, uma das teclas do teclado do meu portátil partiu recentemente e vou precisar de uma substituição:

Categoria classificada:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Gerar Novos Nomes de Produtos
#### Desafio
Criar nomes de produto a partir de palavras exemplares. Aqui incluímos no prompt informação sobre o produto para o qual vamos gerar nomes. Também fornecemos um exemplo semelhante para mostrar o padrão que desejamos receber. Definimos o valor da temperatura alto para aumentar a aleatoriedade e obter respostas mais inovadoras.

Descrição do produto: Um fabricante de batidos caseiro
Palavras-chave: rápido, saudável, compacto.
Nomes de produtos: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Descrição do produto: Um par de sapatos que se adapta a qualquer tamanho de pé.
Palavras-chave: adaptável, ajusta-se, omni-ajuste.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# Referências  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Portal Microsoft Foundry](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Melhores práticas para afinar modelos GPT para classificar texto](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Para Mais Ajuda  
[Equipa de Comercialização da OpenAI](AzureOpenAITeam@microsoft.com) 


# Colaboradores
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:
Este documento foi traduzido utilizando o serviço de tradução automática [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos pela precisão, esteja ciente de que traduções automáticas podem conter erros ou imprecisões. O documento original na sua língua nativa deve ser considerado a fonte autorizada. Para informações críticas, recomenda-se tradução profissional humana. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas resultantes da utilização desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
